# Week 6: Cleaning data


This week we will continue our work with the [Scotiabank Giller Prize](https://scotiabankgiller.com/) nominees.

We will review last week's Python concepts along the way, and now that we have a bit more time, we will start with the original data and consider how best to clean or prepare it for computation.

Review from Week 5:
- Reading a CSV file with the `csv` module
- Dictionaries
- Reading each row as a dictionary with `csv.DictReader`
- Counting with a dictionary

New Python concepts:
- Removing stray spaces with `.strip()`
- Writing a file with `csv.writer`
- Averages and medians
- A reusable counting function
- Data cleaning with a dictionary of corrections

## 1. Review: Loading the data set
The file `giller_nominees.csv` (in the same folder as this notebook) contains one row for every nominated book. For each book we have the year, title, and author, and some biographical information about each author: where they were born, their date of birth, where they live, and so on.

Like all real data sets, the data is not entirely consistent, there is some missing data, and it isn't always in the form that is useful for processing. Part of our task will be to clean the data.

Last week, we showed how to open the file using `open()`, and then learned that there is a module tailored to reading and writing CSV files called, appropriately, `csv`.

In [ ]:
import csv

with open("giller_nominees.csv", encoding="utf-8") as f:
    reader = csv.reader(f)
    rows = list(reader)   # turn all of the rows into one big list

In [ ]:
# review

# How many rows are there in "rows"?
print(len(rows), "rows")
print(rows[0])   # the header row
print(rows[2])

# How many values (columns) are in each row?
print(len(rows[2]), "values")

print(type(rows))
print(type(rows[2]))

# We can get at an individual cell (or field) by assigning a temporary variable # or by indexing rows twice.
print(rows[2][1])

# or
book = rows[2]
print(book[1])

## 2. Cleaning up stray spaces

Look again at the header printed above: `'Year '`, `'Title '`, `'Author '`. Many of the column names have a space at the end, and so do some of the values (`'Toronto, ON '`). The spaces were probably typed by accident or are a result of copy and paste, and they're invisible in a spreadsheet. Do they matter?

To Python, `"Author "` and `"Author"` are **different strings**. Not only will this be super annoying as we write our code, but it will also lead to errors when we start to ask questions about the data such as "how many times was each author nominated?"

The first step in cleaning our data will be to remove these stray spaces that come before or after all the data in the spreadsheet. Here is some code to do that. Spend some time with it to make sure you understand it.

In [ ]:
# clean_rows will hold our cleaned table
clean_rows = []

for row in rows:
    # We will strip white space from each element of row and add it to clean_row
    clean_row = []
    for value in row:
        clean_row.append(value.strip())
        
    clean_rows.append(clean_row)

# Look at one row from the original table and what it looks like after.
# Notice the space after the title of the book is gone.
print(rows[15])
print(clean_rows[15])

The header is row 0, and we can see that the column names were cleaned too:

In [ ]:
print(clean_rows[0])

### Saving the clean data

We don't want to repeat this cleaning every time we use the data, so let's save `clean_rows` as a new CSV file.

Notice that we are **not** changing the original file. This is a good habit in any kind of data work. We would generally prefer to keep the original data as we received it and then the code that we write is a record of what we changed. If the spreadsheet is updated (and ours will be!), we can just run the code again. There are some tradeoffs to make, however. Since we are actively working on developing this data set, it makes sense to fix errors in the original data set. For example, the data set we are using this week has more accurate place names, with correct province abbreviations. We will still discover more issues with the placename data when we try to use it.

Writing a file is a lot like reading one. We `open()` the file in **write mode** with `"w"`, and then use `csv.writer()`. Its `.writerows()` method takes a list of lists, which is the same kind of value `csv.reader` gave us, and writes each inner list as one row.

**Be careful:** opening a file in `"w"` mode erases anything that was already in a file with that name!

In [ ]:
# newline="" stops csv.writer from adding an extra blank line between rows on Windows
with open("giller_nominees_clean.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(clean_rows)

print("Saved giller_nominees_clean.csv")

Open `giller_nominees_clean.csv` in JupyterLab's file browser to check the result. Notice that values containing a comma, like `"Colombo, Sri Lanka"`, were wrapped in quotes automatically: `csv.writer` follows the same rules as `csv.reader`.

## 3. Review: each row as a dictionary with `csv.DictReader`

Last week we used `csv.DictReader()`, which uses the header line as **keys** and turns every row into a dictionary. That lets us write `book["Born"]` instead of having to remember which column number holds the birthplace.

This time we'll read the **clean** file we saved in section 2. (If you get a `FileNotFoundError`, go back and run the cells in section 2 first.)

In [ ]:
with open("giller_nominees_clean.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    nominees = list(reader)

print(len(nominees), "nominated books")   # notice the header is not counted as a book
print(nominees[1])

This makes it more legible to identify different aspects of the data we have about individual books.

In [ ]:
book = nominees[1]
print(book["Author"])
print(f"{book['Title']} by {book['Author']} ({book['Year']})")

Notice that inside the f-string we used single quotes around the key (`book['Title']`). That's because the f-string itself is surrounded by double quotes, and Python would get confused if we used double quotes on the inside too. 

Now we can start to take a look at this data in more interesting ways. Let's print all of the books from one year:

In [ ]:
# for each row of the table stored in nominees
for book in nominees:
    if book["Year"] == "1996":
        print(f"{book['Title']} by {book['Author']}")

## 4. Review: how many times was each author nominated?

Let's revisit last week's question using our clean data. We want a new table with one row per **author**, giving the number of times they were nominated. A dictionary is perfect for this: the **keys** will be author names and the **values** will be counts.

This is one of the most useful patterns and is a variation on a pattern we have already seen, so let's go through it again slowly:
- Start with an empty dictionary.
- For each book, look at its author.
    - If we've seen this author before, add 1 to their count.
    - If not, this is their first nomination, so set their count to 1.

In [ ]:
# create an empty dictionary
nomination_counts = {}

# Go through the rows in the table
for book in nominees:
    # set author to the author of the current row
    author = book["Author"]

    # if we have already started counting nominations for author
    # then we look up their nomination count so far and add one to it
    # otherwise, this is the first nomination we have see for this author
    # so we set its value in the dictionary to 1.
    if author in nomination_counts:
        nomination_counts[author] = nomination_counts[author] + 1
    else:
        nomination_counts[author] = 1

print(len(nomination_counts), "different authors")

In [ ]:
print(nomination_counts["Margaret Atwood"])
print(nomination_counts["Wayne Johnston"])

### Who was nominated the most?

To find the largest count, we can use the same idea as finding the longest word: keep track of the biggest value we've seen so far.

In [ ]:
most_nominations = 0

for author in nomination_counts:
    if nomination_counts[author] > most_nominations:
        most_nominations = nomination_counts[author]

print("The most nominations for any one author:", most_nominations)

for author in nomination_counts:
    if nomination_counts[author] == most_nominations:
        print(author)

### Our new table, from most to least nominated

Let's print every author who was nominated more than once, starting with the most nominated. One way to do that is to count *down* from `most_nominations` with `range()`, and print the authors with that many nominations at each step. (`range(most_nominations, 1, -1)` counts down from `most_nominations` to 2. The `-1` means "count by steps of -1".)

In [ ]:
for count in range(most_nominations, 1, -1):
    for author in nomination_counts:
        if nomination_counts[author] == count:
            print(f"{count}  {author}")

One question that we should file away for later is whether we need to do something special with the author listed as `Kim Thúy, translated by Sheila Fischman`. Should we count that as one author?  Should we split that field into two components?  Are there other authors nominated who might have different translators for each book?  These questions aren't really about programming but are considerations when we think about analyzing the data.

## 5. What is the average number of nominations per author?

The **average** (or **mean**) is the total divided by how many there are. The total number of nominations is the number of books, and the number of authors is the number of keys in our dictionary.

In [ ]:
total_nominations = 0
for author in nomination_counts:
    total_nominations = total_nominations + nomination_counts[author]

number_of_authors = len(nomination_counts)
average = total_nominations / number_of_authors

print(f"{total_nominations} nominations, {number_of_authors} authors")
print(f"On average, an author is nominated {average:.2f} times")

The `:.2f` inside the curly braces tells the f-string to show the number with 2 digits after the decimal point.

An average of about 1.4 might suggest that a typical author is nominated "one and a bit" times. But no one is nominated 1.4 times! Let's look more closely at how the counts are spread out.

In [ ]:
nominated_once = 0
for author in nomination_counts:
    if nomination_counts[author] == 1:
        nominated_once = nominated_once + 1

percent = nominated_once / number_of_authors * 100
print(f"{nominated_once} of {number_of_authors} authors ({percent:.0f}%) were nominated only once")

Most authors are nominated exactly once. A small number of authors with many nominations pull the average up. When data is lopsided like this, the average can be misleading. The **median**, the value in the middle if we lined every author up from fewest nominations to most, is a better description of the "typical" author. Here, the median is 1.

**Discussion:** Is a Giller nomination something that happens to an author once in a career, or does the prize keep returning to a small circle of familiar names? How would you use these numbers to argue either side?

## 6. Saving our new table as a CSV file

We've built a new table of authors and counts. Let's save it as a CSV file so we can open it in a spreadsheet program or use it later.

This time we don't already have a list of lists to give to `.writerows()`, so we use `.writerow()` (no "s"), which writes a single row. We write the header first, and then one row for each author.

In [ ]:
# newline="" stops csv.writer from adding an extra blank line between rows on Windows
with open("author_nominations.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Author", "Nominations"])   # the header row
    for count in range(most_nominations, 0, -1):
        for author in nomination_counts:
            if nomination_counts[author] == count:
                writer.writerow([author, count])

print("Saved author_nominations.csv")

Open `author_nominations.csv` in JupyterLab's file browser to check the result. Any author whose name contains a comma would be wrapped in quotes automatically. `csv.writer` follows the same rules as `csv.reader`.

## 7. A reusable counting function

We are going to want this counting pattern again and again. When we keep writing the same code, it's a sign we should write a **function**. This one counts how many times each value appears in any column we choose.

In [ ]:
def count_values(books, column):
    """ Return a dictionary where the keys are the values found in column,
        and the values are the number of books with that value.
    """
    counts = {}
    for book in books:
        value = book[column]
        if value in counts:
            counts[value] = counts[value] + 1
        else:
            counts[value] = 1
    return counts

Let's use it to count the number of nominated books in each year:

In [ ]:
per_year = count_values(nominees, "Year")

for year in per_year:
    print(year, per_year[year])

**Discussion:** Something changes partway through the list. Before a certain year there are 5 or 6 books per year, and after it there are 12 or more. What might explain that? (Hint: look up how the Giller Prize announces its *longlist* and *shortlist*.) How does this change our interpretation of the "average nominations" we calculated above? Is a longlist nomination the same kind of thing as a shortlist nomination?

The main point of this discussion is to emphasize that it is important to know what is behind the data, and when anomalies show up, that means it's time to go back and learn more about the data.

## 8. Data cleaning: where were the nominees born?

Now let's try to answer a question about geography. The `Born` column holds values like `"Colombo, Sri Lanka"` and `"Petrolia, ON"`: a city followed by a province, state, or country. To ask questions like "how many nominees were born outside Canada?" we need to **split the city from the region**.

### Step 1: One birthplace per author

Our rows are *books*, not *authors*. If we counted birthplaces row by row, an author with 6 nominations would be counted 6 times. So first we build a dictionary that maps each author to their birthplace. Remember that dictionary keys are unique, so each author appears only once, however many books they have.

In our data set, every author has a birthplace entered, but checking for empty values is a good habit: if new rows are added to the spreadsheet before their details are filled in, our code will still work.

In [ ]:
birthplaces = {}

for book in nominees:
    born = book["Born"]
    if born != "":
        birthplaces[book["Author"]] = born

print(len(birthplaces), "authors with a birthplace")
print(birthplaces["Shyam Selvadurai"])

### Step 2: Splitting city from region

`.split(",")` works well here: the `csv` module has already dealt with the quotes, so any comma left inside a value really does separate the city from the region.

In [ ]:
born = birthplaces["Shyam Selvadurai"]
parts = born.split(",")
print(parts)

Notice the space at the start of `' Sri Lanka'`: we'll need `.strip()` again. Also, some birthplaces have more than one comma, and some have none at all:

In [ ]:
print(birthplaces["Joseph Boyden"].split(","))
print(birthplaces["Timothy Taylor"].split(","))

Using index `-1` always gives us the **last** piece, which is the region we want, no matter how many commas there are. And if there's no comma, the whole value is the region.

One more wrinkle: at least one birthplace was typed with a stray comma at the end: `"Lagos, Nigeria,"`. Splitting that gives an empty string as the last piece! To guard against this, we first remove any commas from the ends of `place`. If we give `.strip()` an argument, it removes that character from both ends instead of removing whitespace.

Let's put these steps together with a real value for `place`.

In [ ]:
place = "Lagos, Nigeria,"

# remove stray commas from the ends
place = place.strip(",")   

# split place into a list of parts
parts = place.split(",")

# The region is the last item in the list, and we want to remove
# the whitespace around it.
region = parts[-1].strip()
print(region)

Now that we know how to obtain the region from a birth place, let's make it a function so that we can reuse this code.

In [ ]:
def get_region(place):
    """ Return the province, state or country at the end of place.

        Example: get_region("Colombo, Sri Lanka") returns "Sri Lanka"
    """
    place = place.strip(",")   
    parts = place.split(",")   
    region = parts[-1].strip()
    
    return region


print(get_region("Colombo, Sri Lanka"))
print(get_region("Willowdale, North York, ON"))
print(get_region("Venezuela"))
print(get_region("Lagos, Nigeria,"))

### Step 3: Count the regions, and look at what we got

We can't use `count_values()` directly because we want to count *authors*, not books, so we write the counting loop over the `birthplaces` dictionary instead.

In [ ]:
region_counts = {}

for author in birthplaces:
    region = get_region(birthplaces[author])
    if region in region_counts:
        region_counts[region] = region_counts[region] + 1
    else:
        region_counts[region] = 1

for region in region_counts:
    print(region, region_counts[region])

Read through this list carefully. This is where a computer can't do our thinking for us. What problems do you see?

- **The same place written different ways:** `Ontario` and `ON`, `Alberta` and `AB`, `British Columbia` and `BC`. To Python, each pair is two different regions.
- **Mixed levels:** Canadian provinces appear next to whole countries (`England`, `Kenya`). Most authors born in the United States are recorded as `USA`, but two end with a state abbreviation instead (`MD`, `IL`). Python has no way of knowing that `IL` is a U.S. state while `ON` is a Canadian province; only a person who knows the data can catch this.
- **Missing detail:** one birthplace is just `Canada`. We know the author was born in Canada, but not in which province.
- **Historical names:** `West Germany`, `South Vietnam`, and `Yugoslavia` no longer exist, and the political status of `Tibet` is contested. Should an author born in West Germany in 1960 count as born in "Germany"? There isn't one right answer, but **we** have to decide, and we should say what we decided.
- **Placeholders:** a `-` probably means "unknown."

### Step 4: Fixing values with a dictionary

A dictionary is a handy way to record our cleaning decisions: the key is the messy value, and the value is what we want to replace it with. Writing the decisions down in one place also makes them easy for someone else to check and argue with.

In [ ]:
corrections = {
    # Full province names: use the abbreviations found everywhere else
    "Ontario": "ON",
    "Alberta": "AB",
    "British Columbia": "BC",
    # U.S. states recorded without the country
    "MD": "USA",   # Maryland
    "IL": "USA",   # Illinois
}


def clean_region(region):
    """ Return the corrected version of region, or region unchanged if it
        doesn't need correcting.
    """
    if region in corrections:
        return corrections[region]
    return region


print(clean_region("Ontario"))
print(clean_region("ON"))

Notice what we did **not** put in `corrections`: we have left historical names like `West Germany` exactly as they were recorded. That is a judgment call too, and it doesn't affect the question we are about to ask.

Some problems shouldn't be "fixed" in our code at all. A `-` means the birthplace is unknown, and the right fix is to find the information and enter it in the spreadsheet. In our code, we'll **skip** values we can't use and print them so that we can report them.

In [ ]:
clean_region_counts = {}

for author in birthplaces:
    region = clean_region(get_region(birthplaces[author]))

    if region == "-":
        print(f"Skipping {author}: birthplace unknown")
    elif region in clean_region_counts:
        clean_region_counts[region] = clean_region_counts[region] + 1
    else:
        clean_region_counts[region] = 1

print()
for region in clean_region_counts:
    print(region, clean_region_counts[region])

### Step 5: Born in Canada, or elsewhere?

Finally, let's group the regions into two categories. A list of the Canadian province and territory abbreviations, together with the `in` operator, does the job. We also include `"Canada"` itself, for the author whose birthplace doesn't name a province and "Treaty 6 Territory".  Treaty 6 Territory spans Alberta and Saskatchewan, so we have to make a decision about how to record it.

In [ ]:
canadian_regions = ["AB", "BC", "MB", "NB", "NL", "NS", "NT", "NU", "ON", "PE", "QC", "SK", "YT", "Canada", "Treaty 6 Territory"]

born_in_canada = 0
born_elsewhere = 0

for region in clean_region_counts:
    if region in canadian_regions:
        born_in_canada = born_in_canada + clean_region_counts[region]
    else:
        born_elsewhere = born_elsewhere + clean_region_counts[region]

total = born_in_canada + born_elsewhere
print(f"Born in Canada:    {born_in_canada} ({born_in_canada / total * 100:.0f}%)")
print(f"Born elsewhere:    {born_elsewhere} ({born_elsewhere / total * 100:.0f}%)")

**Discussion:**
- About a quarter of the nominees we have data for were born outside Canada. How might you include this information in a discussion of what "counts" as a Canadian Literature?
- Three authors' birthplaces are unknown (`-`). Could they change the result? More generally, which authors are most likely to have missing biographical information, and why does that matter?
- Every cleaning decision we made (turning `MD` into `USA`, leaving `West Germany` as recorded, skipping unknown values, counting authors rather than books) changes the final numbers a little. Where should we record those decisions so that a reader of our analysis can judge them?

## 9. Try it yourself

1. The `Residence` column records where each author lives now. Use `get_region()` and the counting pattern to find out which province the most nominees live in. Is it the same as the province where the most nominees were born? (Look closely at the regions you get: the `Residence` column needs some cleaning of its own!)
2. Use `count_values()` to find out which column in the spreadsheet is the most complete. (Hint: count how many books have an empty string `""` in each column.)
3. Write a function `books_by(author)` that returns a list of the titles of every nominated book by that author.